In [5]:
from pathlib import Path
from src.settings import AppSettings
from optuna.trial import FixedTrial

settings = AppSettings.from_yaml("./config.yaml")

# Values must fall within the bounds defined in C.HYPERPARAMETERS
params = {
    'max_depth': 6,
    'learning_rate': 1e-3,
    'subsample': 0.7,
    'colsample_bytree': 1.0,
    'min_child_weight': 1,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'gamma': 0.1,
}

mock_trial = FixedTrial(params)



# Derived paths
print(settings.paths.store_file)        # ../datasets/rossmann-store-sales/store.csv

# DateOffset lists
lags  = settings.training.lags.to_date_offsets()
diffs = settings.training.diffs.to_date_offsets()

# Dict ready for xgb.train
xgb_params = settings.xgb.model_dump()

# Optuna search space
for name, spec in settings.hyperparameters.items():
    suggest = getattr(mock_trial, spec.method)
    low  = int(spec.low) if spec.method == "suggest_int" else spec.low
    high = int(spec.high) if spec.method == "suggest_int" else spec.high
    value = suggest(name, low, high, log=spec.log)

../datasets/rossmann-store-sales\store.csv


In [ ]:
.git/

AppSettings(paths=PathsSettings(data_dir='../datasets/rossmann-store-sales', log_dir='./artifacts', store_file='../datasets/rossmann-store-sales\\store.csv', train_file='../datasets/rossmann-store-sales\\train.csv', log_file='./artifacts\\hypertuning.log', storage_url='sqlite:///./artifacts\\hypertuning.db'), training=TrainingSettings(forecast_horizon=10, roll_windows=['7D', '14D', '30D', '60D'], diffs=DiffOffsets(days=[1, 2, 3, 4, 5, 6, 14, 30]), lags=LagOffsets(days=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], weeks=[3, 4], months=[2, 3, 4, 5, 6], years=[1])), cv=CVSettings(n_outer_splits=5, n_inner_splits=3), hypertuning=HypertuningSettings(num_trials=100, seed=42, monitor_periods=100, num_startup_trials=5, num_jobs=-1, early_stopping_rounds=10, num_boost_rounds=10000, refit_val_fraction=0.1), xgb=XGBSettings(tree_method='hist', device='cpu', objective='reg:squarederror', eval_metric='rmse', verbosity=0), hyperparameters={'max_depth': HyperparameterSpec(method='suggest_int', low